# Models

> Core data models for file discovery including FileInfo, FileType, and DirectoryInfo.

In [ ]:
#| default_exp core.models

In [ ]:
#| export
from enum import Enum
from dataclasses import dataclass, field
from datetime import datetime
from typing import Any, Dict, Optional

from cjm_file_discovery.utils.formatting import format_file_size, format_timestamp

## FileType Enum

File type categories for categorizing discovered files.

In [ ]:
#| export
class FileType(str, Enum):
    """File type categories."""
    AUDIO = "audio"
    VIDEO = "video"
    IMAGE = "image"
    DOCUMENT = "document"
    CODE = "code"
    DATA = "data"        # .json, .xml, .csv, .db
    ARCHIVE = "archive"  # .zip, .tar, .gz
    OTHER = "other"

In [ ]:
# Test FileType enum
assert FileType.AUDIO.value == "audio"
assert FileType.VIDEO == "video"  # str enum comparison
assert str(FileType.IMAGE.value) == "image"
print("FileType enum tests passed!")

FileType enum tests passed!


## FileInfo

Metadata for a discovered file or directory.

In [ ]:
#| export
@dataclass
class FileInfo:
    """Metadata for a discovered file or directory."""
    name: str              # File name with extension
    path: str              # Full path (provider-specific format)
    is_directory: bool     # True for directories
    size: Optional[int] = None              # Size in bytes
    modified: Optional[float] = None        # Last modified timestamp (Unix)
    created: Optional[float] = None         # Creation timestamp (if available)
    file_type: FileType = FileType.OTHER    # Categorized file type
    extension: Optional[str] = None         # File extension (without dot)
    mime_type: Optional[str] = None         # MIME type (if determinable)
    provider_name: str = "local"            # Source provider identifier
    metadata: Dict[str, Any] = field(default_factory=dict)  # Provider-specific extras

    @property
    def size_str(self) -> str:  # Human-readable size string
        """Human-readable size string (e.g., '15.2 MB')."""
        if self.size is None:
            return ""
        return format_file_size(self.size)

    @property
    def modified_str(self) -> str:  # Human-readable modified date
        """Human-readable modified date."""
        if self.modified is None:
            return ""
        return format_timestamp(self.modified)

In [ ]:
import time

# Test FileInfo
file_info = FileInfo(
    name="test.mp3",
    path="/path/to/test.mp3",
    is_directory=False,
    size=1572864,
    modified=time.time(),
    file_type=FileType.AUDIO,
    extension="mp3"
)

assert file_info.name == "test.mp3"
assert file_info.is_directory == False
assert file_info.size_str == "1.5 MB"
assert file_info.modified_str == "Just now"
assert file_info.file_type == FileType.AUDIO

# Test directory
dir_info = FileInfo(
    name="folder",
    path="/path/to/folder",
    is_directory=True
)
assert dir_info.is_directory == True
assert dir_info.size_str == ""  # No size for directories by default

print("FileInfo tests passed!")

FileInfo tests passed!


## DirectoryInfo

Metadata for a directory with optional computed statistics.

In [ ]:
#| export
@dataclass
class DirectoryInfo:
    """Metadata for a directory with optional computed statistics."""
    path: str                                  # Full directory path
    name: str                                  # Directory name
    item_count: Optional[int] = None           # Number of direct children
    total_size: Optional[int] = None           # Total size of contents (bytes)
    file_count: Optional[int] = None           # Number of files (recursive)
    directory_count: Optional[int] = None      # Number of subdirectories

    @property
    def total_size_str(self) -> str:  # Human-readable total size
        """Human-readable total size string."""
        if self.total_size is None:
            return ""
        return format_file_size(self.total_size)

In [ ]:
# Test DirectoryInfo
dir_info = DirectoryInfo(
    path="/path/to/folder",
    name="folder",
    item_count=10,
    total_size=1073741824,  # 1 GB
    file_count=50,
    directory_count=5
)

assert dir_info.name == "folder"
assert dir_info.item_count == 10
assert dir_info.total_size_str == "1.0 GB"
assert dir_info.file_count == 50

# Test without stats
dir_info_minimal = DirectoryInfo(path="/path", name="path")
assert dir_info_minimal.total_size_str == ""

print("DirectoryInfo tests passed!")

DirectoryInfo tests passed!


In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()